# BAW Notebook 02: Antibody Backbone Design (RFantibody)

**Purpose:** Design *de novo* antibody/nanobody backbones targeting a user-defined epitope
using RFantibody (Baker Lab, Nature 2025). Runs the full three-step pipeline:

1. **RFdiffusion** — generates antibody backbone structures docked to the target epitope
2. **ProteinMPNN** — designs CDR loop amino acid sequences for each backbone
3. **RF2** — predicts the final folded structure and scores each design (pAE, RMSD)

**Input:** `hotspot_residues.json` and cleaned target PDB from Notebook 01

**Output:**
- Quiver files for each pipeline stage (`1_rfdiffusion.qv`, `2_proteinmpnn.qv`, `3_rf2.qv`)
- Filtered score table (`rf2_scores_filtered.csv`)
- Top design PDB files ready for Notebook 03
- Design summary JSON for Notebook 03

**Framework note:** Notebook 02 designs nanobody-format binders (VHH, single heavy chain)
using the h-NbBCII10 framework bundled with RFantibody. Full IgG format is supported
by selecting the ScFv framework and adjusting CDR loops in Cell 2.

**Reference:** Bennett et al., Nature (2025). `github.com/RosettaCommons/RFantibody` (MIT)


In [ ]:
# Cell 1: Imports

import os
import sys
import json
import subprocess
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB import NeighborSearch

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import tkinter as tk
from tkinter import filedialog, messagebox

warnings.filterwarnings('ignore')

# --------------------------------------------------------------------------
# Popup helpers
# --------------------------------------------------------------------------
def get_tk_root():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    return root

def browse_file(title='Select file', filetypes=None):
    if filetypes is None:
        filetypes = [('All files', '*.*')]
    root = get_tk_root()
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    return path

def browse_directory(title='Select folder'):
    root = get_tk_root()
    path = filedialog.askdirectory(title=title)
    root.destroy()
    return path

# --------------------------------------------------------------------------
# Subprocess helper: run command, stream stdout, raise on failure
# --------------------------------------------------------------------------
def run_cmd(cmd, cwd=None, env=None, label=''):
    """Run a shell command, print output in real time, raise on error."""
    print(f'\n>>> {label}' if label else f'\n>>> Running: {" ".join(str(c) for c in cmd)}')
    print('-' * 60)
    proc = subprocess.Popen(
        [str(c) for c in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(cwd) if cwd else None,
        env=env,
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print('-' * 60)
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc.returncode

print('Imports loaded.')


In [ ]:
# Cell 2: Configuration
# =============================================================
# Edit these values before running the rest of the notebook.
# =============================================================

# ---  RFantibody repository location  ---
# Where to clone (or find) the RFantibody repo.
RFANTIBODY_DIR = Path.home() / 'RFantibody'

# ---  Antibody format  ---
# 'nanobody' : single heavy chain VHH (default, simpler, more characterized)
# 'scfv'     : single-chain variable fragment (heavy + light chains)
ANTIBODY_FORMAT = 'nanobody'

# ---  CDR loop lengths  ---
# Format: "loop:length" or "loop:min-max"
# Nanobody (H1, H2, H3 only):
CDR_LOOPS_NANOBODY = "H1:7,H2:6,H3:5-13"
# Full scFv (heavy + light):
CDR_LOOPS_SCFV     = "H1:7,H2:6,H3:5-13,L1:8-13,L2:7,L3:9-11"

# ---  Design scale  ---
N_DESIGNS_RFD     = 50    # RFdiffusion backbones (use 50-100 for pilot, 1000+ for production)
MPNN_SEQS_PER_STRUCT = 4  # ProteinMPNN sequences per backbone
RF2_RECYCLES      = 10    # RF2 recycling iterations (10 recommended)

# ---  Target cropping  ---
# Residues within CROP_RADIUS of the epitope COM are kept.
# Strongly recommended: reduces compute and avoids irrelevant structure.
CROP_TARGET       = True
CROP_RADIUS       = 30.0  # Angstroms around the epitope COM

# ---  Filtering thresholds  ---
FILTER_PAE_MAX    = 10.0  # RF2 pAE < this (lower = better)
FILTER_RMSD_MAX   =  2.0  # Design vs RF2-predicted RMSD < this (Angstroms)

# ---  Top designs to export for Notebook 03  ---
N_TOP_DESIGNS     = 10

# ---  Deterministic mode (reproducible results)  ---
DETERMINISTIC     = False  # True for debugging; False for diversity

# Derived: pick the right CDR loop string
CDR_LOOPS = CDR_LOOPS_NANOBODY if ANTIBODY_FORMAT == 'nanobody' else CDR_LOOPS_SCFV

print('Configuration loaded.')
print(f'  Format           : {ANTIBODY_FORMAT}')
print(f'  CDR loops        : {CDR_LOOPS}')
print(f'  RFdiffusion N    : {N_DESIGNS_RFD}')
print(f'  ProteinMPNN seqs : {MPNN_SEQS_PER_STRUCT} per backbone')
print(f'  RF2 recycles     : {RF2_RECYCLES}')
print(f'  Target crop      : {CROP_TARGET} ({CROP_RADIUS} A)')
print(f'  Filter pAE < {FILTER_PAE_MAX}, RMSD < {FILTER_RMSD_MAX}')


In [ ]:
# Cell 3: File selection

out = widgets.Output()
display(out)

# --- Load hotspot JSON from Notebook 01 ---
with out:
    print('Select the hotspot_residues.json file from Notebook 01...')

hotspot_json_path = browse_file(
    title='Select hotspot_residues.json (from Notebook 01)',
    filetypes=[('JSON files', '*.json'), ('All files', '*.*')]
)

with out:
    if hotspot_json_path:
        print(f'  Hotspot JSON: {hotspot_json_path}')
    else:
        raise FileNotFoundError('No hotspot JSON selected. Re-run this cell.')

# --- Output directory for this notebook ---
with out:
    print('\nSelect output directory for Notebook 02 results...')

output_dir_str = browse_directory(title='Select output directory (e.g. ~/BasementAntibodyWorks/designs)')

with out:
    if output_dir_str:
        output_dir = Path(output_dir_str)
    else:
        output_dir = Path.home() / 'BasementAntibodyWorks' / 'designs'
        print(f'  (using default: {output_dir})')

output_dir.mkdir(parents=True, exist_ok=True)

with out:
    print(f'  Output dir: {output_dir}')
    print('\nAll paths set.')


In [ ]:
# Cell 4: Load Notebook 01 outputs

with open(hotspot_json_path) as f:
    hotspot_data = json.load(f)

target_name       = hotspot_data['target_name']
source_pdb_name   = hotspot_data['source_pdb']
target_chain      = hotspot_data['chain']
hotspot_res_nums  = hotspot_data['hotspot_residues']
epitope_com       = np.array(hotspot_data['epitope_com'])
clean_pdb_path    = Path(hotspot_data['clean_pdb'])

print(f'NOTEBOOK 01 OUTPUTS LOADED')
print('=' * 60)
print(f'Target name       : {target_name}')
print(f'Source PDB        : {source_pdb_name}')
print(f'Chain             : {target_chain}')
print(f'Hotspot residues  : {hotspot_res_nums}')
print(f'Epitope COM       : ({epitope_com[0]:.2f}, {epitope_com[1]:.2f}, {epitope_com[2]:.2f})')
print(f'Clean PDB         : {clean_pdb_path}')
print()

if not clean_pdb_path.exists():
    raise FileNotFoundError(
        f'Clean PDB not found at {clean_pdb_path}\n'
        f'Run Notebook 01 first, or update the path in hotspot_residues.json.'
    )

# Build hotspot string for RFantibody: e.g. "A35,A119,A245,A342"
hotspot_str = ','.join(f'{target_chain}{r}' for r in hotspot_res_nums)
print(f'RFantibody hotspot string: {hotspot_str}')


In [ ]:
# Cell 5: Crop target PDB to the epitope region
#
# RFdiffusion and RF2 scale as O(N^2) with residue count.
# Cropping to the region around the epitope is strongly recommended.
# Keeps all residues within CROP_RADIUS of the epitope COM.

parser = PDBParser(QUIET=True)
structure = parser.get_structure('target', str(clean_pdb_path))
model     = structure[0]
chain     = model[target_chain]

if CROP_TARGET:
    # Collect protein residues that have a CA atom (skip any with missing/disordered CA)
    ca_residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    ca_coords   = np.array([r['CA'].get_vector().get_array() for r in ca_residues])

    # Find residues within CROP_RADIUS of epitope COM
    dists_to_com = np.linalg.norm(ca_coords - epitope_com, axis=1)
    keep_mask    = dists_to_com <= CROP_RADIUS

    protein_res  = [r for r in chain if r.get_id()[0] == ' ']
    keep_res_ids = set(
        ca_residues[i].get_id()[1]
        for i, keep in enumerate(keep_mask)
        if keep
    )

    print(f'TARGET CROPPING')
    print('=' * 60)
    print(f'Original residues : {len(protein_res)}')
    print(f'Residues within {CROP_RADIUS} A of epitope COM: {len(keep_res_ids)}')
    print(f'Residue range kept: {min(keep_res_ids)} — {max(keep_res_ids)}')
    
    # Verify all hotspot residues are included
    missing_hotspots = [r for r in hotspot_res_nums if r not in keep_res_ids]
    if missing_hotspots:
        print(f'WARNING: hotspot residues not in crop: {missing_hotspots}')
        print(f'Increase CROP_RADIUS in Cell 2 to include them.')
    else:
        print(f'All {len(hotspot_res_nums)} hotspot residues are within the crop.')

    class CropSelect(Select):
        def accept_chain(self, c):
            return c.get_id() == target_chain
        def accept_residue(self, r):
            return r.get_id()[0] == ' ' and r.get_id()[1] in keep_res_ids
        def accept_atom(self, a):
            altloc = a.get_altloc()
            return altloc == ' ' or altloc == 'A'

    target_for_design = output_dir / f'{clean_pdb_path.stem}_cropped.pdb'
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(target_for_design), CropSelect())
    print(f'\nCropped target saved: {target_for_design}')

else:
    target_for_design = clean_pdb_path
    protein_res = [r for r in chain if r.get_id()[0] == ' ']
    print(f'Cropping disabled. Using full target: {len(protein_res)} residues.')
    print(f'NOTE: large targets slow down RFdiffusion significantly.')


In [ ]:
# Cell 6: Check and install RFantibody
#
# RFantibody uses its own uv-managed Python environment (.venv).
# It is installed separately from the BAW conda environment.
# All pipeline commands are called via subprocess.

rfab_dir     = Path(RFANTIBODY_DIR)
weights_dir  = rfab_dir / 'weights'
venv_dir     = rfab_dir / '.venv'

print('RFANTIBODY INSTALLATION CHECK')
print('=' * 60)

# ---- 1. Clone repo if missing ----
if not rfab_dir.exists():
    print(f'Cloning RFantibody to {rfab_dir}...')
    run_cmd(
        ['git', 'clone', 'https://github.com/RosettaCommons/RFantibody.git', str(rfab_dir)],
        label='git clone RFantibody'
    )
else:
    print(f'Repo found: {rfab_dir}')

# ---- 2. Install uv if missing ----
uv_path = shutil.which('uv')
if uv_path is None:
    print('\nuv not found. Installing...')
    run_cmd(
        ['bash', '-c', 'curl -LsSf https://astral.sh/uv/install.sh | sh'],
        label='install uv'
    )
    # Add uv to PATH for this session
    uv_bin = Path.home() / '.local' / 'bin'
    os.environ['PATH'] = str(uv_bin) + ':' + os.environ.get('PATH', '')
    uv_path = shutil.which('uv')
    if uv_path is None:
        raise RuntimeError(f'uv not found after install. Open a new terminal and re-run.')
else:
    print(f'uv found: {uv_path}')

# ---- 3. Download weights if missing ----
if not weights_dir.exists() or not any(weights_dir.glob('*.pt')):
    print(f'\nWeights not found. Downloading (~3 GB, this takes a while)...')
    run_cmd(
        ['bash', 'include/download_weights.sh'],
        cwd=rfab_dir,
        label='download RFantibody weights'
    )
else:
    weight_files = list(weights_dir.glob('*.pt'))
    print(f'Weights found: {len(weight_files)} .pt file(s) in {weights_dir}')

# ---- 4. Set up venv if missing ----
if not venv_dir.exists():
    print(f'\nSetting up Python environment (uv sync)...')
    run_cmd(['uv', 'sync'], cwd=rfab_dir, label='uv sync')
else:
    print(f'venv found: {venv_dir}')

# ---- 5. Verify installation ----
print('\nVerifying rfdiffusion --help...')
result = subprocess.run(
    ['uv', 'run', 'rfdiffusion', '--help'],
    cwd=str(rfab_dir),
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print('rfdiffusion: OK')
else:
    print('rfdiffusion --help output:')
    print(result.stdout[:500])
    print(result.stderr[:500])
    raise RuntimeError('rfdiffusion not working. Check installation above.')

print()
print('RFantibody installation verified.')

# ---- Helper: build uv run command from rfab_dir ----
def rfab_cmd(tool, *args):
    """Return a command list to run a RFantibody tool via uv."""
    return ['uv', 'run', tool] + list(args)


In [ ]:
# Cell 7: Select antibody framework
#
# RFantibody requires an HLT-formatted framework PDB as the antibody scaffold.
# The RFantibody repo ships with two ready-to-use frameworks:
#   Nanobody : RFantibody/scripts/examples/example_inputs/h-NbBCII10.pdb
#   ScFv     : RFantibody/scripts/examples/example_inputs/hu-4D5-8_Fv.pdb
#
# You can also download additional frameworks from SAbDab (opig.stats.ox.ac.uk)
# and convert them with: python scripts/util/chothia_to_HLT.py -inpdb in.pdb -outpdb out.pdb

BUNDLED_FRAMEWORKS = {
    'nanobody': rfab_dir / 'scripts' / 'examples' / 'example_inputs' / 'h-NbBCII10.pdb',
    'scfv':     rfab_dir / 'scripts' / 'examples' / 'example_inputs' / 'hu-4D5-8_Fv.pdb',
}

default_framework = BUNDLED_FRAMEWORKS.get(ANTIBODY_FORMAT)

print(f'FRAMEWORK SELECTION')
print('=' * 60)
print(f'Format: {ANTIBODY_FORMAT}')
print(f'Default framework: {default_framework}')

use_default = True  # Set to False and run browse_file() below to use a custom framework

if use_default:
    framework_pdb = default_framework
    if not framework_pdb.exists():
        raise FileNotFoundError(
            f'Bundled framework not found at {framework_pdb}\n'
            f'Verify RFantibody is installed correctly (Cell 6).'
        )
    print(f'Using bundled framework: {framework_pdb}')
else:
    # Uncomment to browse for a custom framework:
    # framework_pdb = Path(browse_file(
    #     title='Select HLT-formatted antibody framework PDB',
    #     filetypes=[('PDB files', '*.pdb')]
    # ))
    pass

print(f'\nFramework: {framework_pdb.name}')
print(f'CDR loops : {CDR_LOOPS}')


In [ ]:
# Cell 8: Run RFdiffusion — backbone design
#
# Generates N_DESIGNS_RFD antibody backbone structures docked to the target,
# with CDR loops positioned to contact the hotspot residues.
#
# Output: 1_rfdiffusion.qv (Quiver file containing all backbone PDBs)
#
# Runtime: ~2-5 min for 10 designs on RTX 4070.
# Checkpoint: skips if 1_rfdiffusion.qv already exists.

rfd_output_qv = output_dir / '1_rfdiffusion.qv'

if rfd_output_qv.exists():
    print(f'Checkpoint: {rfd_output_qv.name} already exists. Skipping RFdiffusion.')
    print(f'Delete {rfd_output_qv} to re-run.')
else:
    cmd = rfab_cmd(
        'rfdiffusion',
        '--target',        str(target_for_design),
        '--framework',     str(framework_pdb),
        '--output-quiver', str(rfd_output_qv),
        '--num-designs',   str(N_DESIGNS_RFD),
        '--design-loops',  CDR_LOOPS,
        '--hotspots',      hotspot_str,
    )
    if DETERMINISTIC:
        cmd.append('--deterministic')

    print(f'RFDIFFUSION — BACKBONE DESIGN')
    print('=' * 60)
    print(f'Target     : {target_for_design.name}')
    print(f'Framework  : {framework_pdb.name}')
    print(f'Hotspots   : {hotspot_str}')
    print(f'CDR loops  : {CDR_LOOPS}')
    print(f'N designs  : {N_DESIGNS_RFD}')
    print(f'Output     : {rfd_output_qv}')
    print()

    run_cmd(cmd, cwd=rfab_dir, label='rfdiffusion backbone design')

# Verify output and report count
result = subprocess.run(
    rfab_cmd('qvls', str(rfd_output_qv)),
    cwd=str(rfab_dir),
    capture_output=True, text=True
)
n_backbones = len(result.stdout.strip().split('\n')) if result.stdout.strip() else 0
print(f'\nBackbones in Quiver: {n_backbones}')


In [ ]:
# Cell 9: Run ProteinMPNN — CDR sequence design
#
# Takes the RFdiffusion backbone Quiver and designs amino acid sequences
# for all CDR loops. Generates MPNN_SEQS_PER_STRUCT sequences per backbone.
# Total sequences = N_DESIGNS_RFD × MPNN_SEQS_PER_STRUCT
#
# Output: 2_proteinmpnn.qv
#
# Checkpoint: skips if output already exists.

mpnn_output_qv = output_dir / '2_proteinmpnn.qv'

if mpnn_output_qv.exists():
    print(f'Checkpoint: {mpnn_output_qv.name} already exists. Skipping ProteinMPNN.')
    print(f'Delete {mpnn_output_qv} to re-run.')
else:
    cmd = rfab_cmd(
        'proteinmpnn',
        '--input-quiver',   str(rfd_output_qv),
        '--output-quiver',  str(mpnn_output_qv),
        '--seqs-per-struct', str(MPNN_SEQS_PER_STRUCT),
        '--temperature',    '0.2',
    )
    if DETERMINISTIC:
        cmd.append('--deterministic')

    print('PROTEINMPNN — CDR SEQUENCE DESIGN')
    print('=' * 60)
    print(f'Input        : {rfd_output_qv.name}')
    print(f'Output       : {mpnn_output_qv.name}')
    print(f'Seqs/struct  : {MPNN_SEQS_PER_STRUCT}')
    print(f'Temperature  : 0.2')
    expected_total = n_backbones * MPNN_SEQS_PER_STRUCT
    print(f'Expected total sequences: {expected_total}')
    print()

    run_cmd(cmd, cwd=rfab_dir, label='proteinmpnn sequence design')

result = subprocess.run(
    rfab_cmd('qvls', str(mpnn_output_qv)),
    cwd=str(rfab_dir),
    capture_output=True, text=True
)
n_sequences = len(result.stdout.strip().split('\n')) if result.stdout.strip() else 0
print(f'\nSequences in Quiver: {n_sequences}')


In [ ]:
# Cell 10: Run RF2 — structure prediction and scoring
#
# RF2 (antibody-finetuned RoseTTAFold2) predicts the structure of each
# designed sequence and scores confidence. Key output metrics:
#   pAE   — predicted aligned error at the antibody-target interface (lower = better)
#   RMSD  — backbone RMSD between the designed structure and the RF2 prediction
#
# Recommended filter: pAE < 10 AND RMSD < 2 Å
#
# Runtime: ~1-2 min per sequence on RTX 4070.
# For n_sequences = 200: ~3-6 hours. Plan as an overnight run.
# Checkpoint: skips if output already exists.

rf2_output_qv = output_dir / '3_rf2.qv'

if rf2_output_qv.exists():
    print(f'Checkpoint: {rf2_output_qv.name} already exists. Skipping RF2.')
    print(f'Delete {rf2_output_qv} to re-run.')
else:
    cmd = rfab_cmd(
        'rf2',
        '--input-quiver',   str(mpnn_output_qv),
        '--output-quiver',  str(rf2_output_qv),
        '--num-recycles',   str(RF2_RECYCLES),
    )

    print('RF2 — STRUCTURE PREDICTION AND SCORING')
    print('=' * 60)
    print(f'Input          : {mpnn_output_qv.name}')
    print(f'Output         : {rf2_output_qv.name}')
    print(f'Recycles       : {RF2_RECYCLES}')
    print(f'Total to score : {n_sequences}')
    print()
    print('NOTE: this is the slow step. Budget ~1-2 min per sequence on RTX 4070.')
    mins_estimate = n_sequences * 1.5
    print(f'Estimated runtime: {mins_estimate:.0f} min ({mins_estimate/60:.1f} hours)')
    print('Tip: run this as an overnight job.')
    print()

    run_cmd(cmd, cwd=rfab_dir, label='rf2 structure prediction')

result = subprocess.run(
    rfab_cmd('qvls', str(rf2_output_qv)),
    cwd=str(rfab_dir),
    capture_output=True, text=True
)
n_predicted = len(result.stdout.strip().split('\n')) if result.stdout.strip() else 0
print(f'\nPredicted structures in Quiver: {n_predicted}')


In [ ]:
# Cell 11: Parse RF2 scores and filter designs
#
# Extracts the score table from the RF2 Quiver and applies filtering thresholds.
# Also extracts the RFdiffusion hotspot recovery scores.

rf2_scorefile  = output_dir / '3_rf2.sc'
rfd_scorefile  = output_dir / '1_rfdiffusion.sc'

# Extract RF2 scores
run_cmd(
    rfab_cmd('qvscorefile', str(rf2_output_qv)),
    cwd=rfab_dir,
    label='extract RF2 scores'
)
# qvscorefile writes to <quiver_stem>.sc by default
default_rf2_sc = rfab_dir / '3_rf2.sc'
if default_rf2_sc.exists() and not rf2_scorefile.exists():
    shutil.copy(default_rf2_sc, rf2_scorefile)

# Extract RFdiffusion hotspot recovery scores
run_cmd(
    rfab_cmd('qvscorefile', str(rfd_output_qv)),
    cwd=rfab_dir,
    label='extract RFdiffusion scores'
)
default_rfd_sc = rfab_dir / '1_rfdiffusion.sc'
if default_rfd_sc.exists() and not rfd_scorefile.exists():
    shutil.copy(default_rfd_sc, rfd_scorefile)

# ---- Load and inspect RF2 scores ----
if rf2_scorefile.exists():
    scores_df = pd.read_csv(rf2_scorefile, sep='\t')
    print(f'RF2 SCORES')
    print('=' * 60)
    print(f'Total designs scored: {len(scores_df)}')
    print(f'Columns: {list(scores_df.columns)}')
    print()
    display(scores_df.describe())
else:
    # Try to find any .sc file produced
    sc_files = list(output_dir.glob('*.sc')) + list(rfab_dir.glob('3_rf2*.sc'))
    if sc_files:
        scores_df = pd.read_csv(sc_files[0], sep='\t')
        rf2_scorefile = sc_files[0]
        print(f'Loaded scores from: {sc_files[0]}')
        display(scores_df.head())
    else:
        print('WARNING: No score file found. Check the RF2 run output above.')
        scores_df = pd.DataFrame()


In [ ]:
# Cell 12: Apply filters and rank designs

if scores_df.empty:
    print('No scores to filter. Check Cell 11.')
else:
    # Identify pAE and RMSD columns
    # Column names can vary by RFantibody version; find them robustly
    pae_col  = next((c for c in scores_df.columns if 'pae'  in c.lower()), None)
    rmsd_col = next((c for c in scores_df.columns if 'rmsd' in c.lower()), None)
    tag_col  = next((c for c in scores_df.columns if 'tag'  in c.lower() or 
                     'description' in c.lower() or 'name' in c.lower()), None)

    print(f'Score columns identified:')
    print(f'  pAE column   : {pae_col}')
    print(f'  RMSD column  : {rmsd_col}')
    print(f'  Tag column   : {tag_col}')
    print()

    # Apply filters
    filtered = scores_df.copy()
    if pae_col:
        filtered = filtered[filtered[pae_col] < FILTER_PAE_MAX]
    if rmsd_col:
        filtered = filtered[filtered[rmsd_col] < FILTER_RMSD_MAX]

    print(f'FILTERING RESULTS')
    print('=' * 60)
    print(f'Before filter : {len(scores_df)} designs')
    print(f'After filter  : {len(filtered)} designs')
    print(f'  pAE < {FILTER_PAE_MAX}  AND  RMSD < {FILTER_RMSD_MAX} A')
    print()

    # Rank by pAE (lower is better)
    if pae_col and len(filtered) > 0:
        filtered = filtered.sort_values(pae_col).reset_index(drop=True)
        filtered['rank'] = filtered.index + 1

        print(f'TOP {min(N_TOP_DESIGNS, len(filtered))} DESIGNS BY pAE:')
        display_cols = ['rank']
        if tag_col:  display_cols.append(tag_col)
        if pae_col:  display_cols.append(pae_col)
        if rmsd_col: display_cols.append(rmsd_col)
        display(filtered[display_cols].head(N_TOP_DESIGNS))
    elif len(filtered) == 0:
        print('No designs passed the filter.')
        print('Consider relaxing FILTER_PAE_MAX or FILTER_RMSD_MAX in Cell 2.')
        # `filtered` stays empty on purpose: these designs did NOT pass the
        # quality thresholds and must not be exported as "top designs" or
        # counted as passing in Cell 13 / the summary JSON.
        preview_df = scores_df.copy()
        if pae_col:
            preview_df = preview_df.sort_values(pae_col)
        print(f'Showing top 5 unfiltered designs for inspection (none passed filter):')
        display(preview_df.head(5))

    # Save filtered scores
    filtered_csv = output_dir / 'rf2_scores_filtered.csv'
    filtered.to_csv(filtered_csv, index=False)
    print(f'\nFiltered scores saved: {filtered_csv}')


In [ ]:
# Cell 13: Export top designs for Notebook 03

top_designs_dir = output_dir / 'top_designs'
top_designs_dir.mkdir(exist_ok=True)

if not scores_df.empty and tag_col and len(filtered) > 0:
    top_n = min(N_TOP_DESIGNS, len(filtered))
    top_tags = filtered.head(top_n)[tag_col].tolist() if tag_col else []

    print(f'EXPORTING TOP {top_n} DESIGNS')
    print('=' * 60)

    if top_tags:
        # Extract specific designs from RF2 Quiver
        extract_cmd = rfab_cmd('qvextract', str(rf2_output_qv), '-o', str(top_designs_dir))
        run_cmd(extract_cmd, cwd=rfab_dir, label='extract top design PDBs')

        # qvextract dumps every design in the quiver; keep only the top-ranked ones
        top_tag_set = set(top_tags)
        for pdb_file in top_designs_dir.glob('*.pdb'):
            if pdb_file.stem not in top_tag_set:
                pdb_file.unlink()

        all_extracted = list(top_designs_dir.glob('*.pdb'))
        print(f'Extracted {len(all_extracted)} PDB files')

    # Save design summary JSON for Notebook 03
    summary = {
        'target_name':      target_name,
        'target_chain':     target_chain,
        'clean_pdb':        str(clean_pdb_path),
        'cropped_pdb':      str(target_for_design),
        'hotspot_residues': hotspot_res_nums,
        'hotspot_str':      hotspot_str,
        'framework':        str(framework_pdb),
        'antibody_format':  ANTIBODY_FORMAT,
        'cdr_loops':        CDR_LOOPS,
        'n_designs_rfd':    N_DESIGNS_RFD,
        'n_sequences':      n_sequences,
        'n_passed_filter':  len(filtered),
        'n_top_exported':   top_n,
        'filter_pae_max':   FILTER_PAE_MAX,
        'filter_rmsd_max':  FILTER_RMSD_MAX,
        'rf2_scorefile':    str(filtered_csv),
        'top_designs_dir':  str(top_designs_dir),
        'quiver_rf2':       str(rf2_output_qv),
    }

    summary_json = output_dir / 'nb02_design_summary.json'
    with open(summary_json, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'Design summary JSON: {summary_json}')

print()
print('=' * 60)
print(f'NOTEBOOK 02 COMPLETE — {target_name}')
print('=' * 60)
print()
print(f'  RFdiffusion backbones : {n_backbones}')
print(f'  ProteinMPNN sequences : {n_sequences}')
print(f'  RF2 predictions       : {n_predicted}')
print(f'  Passed filter         : {len(filtered) if not scores_df.empty else "N/A"}')
print(f'  Top designs exported  : {min(N_TOP_DESIGNS, len(filtered)) if not scores_df.empty else "N/A"}')
print()
print('Next: Notebook 03 uses nb02_design_summary.json to run additional')
print('sequence optimisation and humanisation with ProteinMPNN.')


In [ ]:
# Cell 14: Visualize score distributions

if not scores_df.empty and (pae_col or rmsd_col):
    fig, axes = plt.subplots(1, 2 if (pae_col and rmsd_col) else 1, figsize=(12, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    ax_idx = 0

    if pae_col:
        ax = axes[ax_idx]; ax_idx += 1
        ax.hist(scores_df[pae_col].dropna(), bins=30, color='steelblue', alpha=0.7, label='all')
        if len(filtered) > 0:
            ax.hist(filtered[pae_col].dropna(), bins=30, color='orange', alpha=0.7, label='passed filter')
        ax.axvline(FILTER_PAE_MAX, color='red', linestyle='--', label=f'cutoff ({FILTER_PAE_MAX})')
        ax.set_xlabel('RF2 pAE (interface)')
        ax.set_ylabel('Count')
        ax.set_title('pAE distribution')
        ax.legend()

    if rmsd_col and ax_idx < len(axes):
        ax = axes[ax_idx]
        ax.hist(scores_df[rmsd_col].dropna(), bins=30, color='steelblue', alpha=0.7, label='all')
        if len(filtered) > 0:
            ax.hist(filtered[rmsd_col].dropna(), bins=30, color='orange', alpha=0.7, label='passed filter')
        ax.axvline(FILTER_RMSD_MAX, color='red', linestyle='--', label=f'cutoff ({FILTER_RMSD_MAX} Å)')
        ax.set_xlabel('RMSD design vs RF2 prediction (Å)')
        ax.set_ylabel('Count')
        ax.set_title('RMSD distribution')
        ax.legend()

    plt.suptitle(f'{target_name} — RFantibody Design Scores', fontsize=13, fontweight='bold')
    plt.tight_layout()

    fig_path = output_dir / 'nb02_score_distributions.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')

    # pAE vs RMSD scatter (if both available)
    if pae_col and rmsd_col and len(scores_df) > 1:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(scores_df[rmsd_col], scores_df[pae_col],
                   c='steelblue', alpha=0.5, s=20, label='all designs')
        if len(filtered) > 0:
            ax.scatter(filtered[rmsd_col], filtered[pae_col],
                       c='orange', alpha=0.8, s=40, label='passed filter', zorder=5)
        ax.axvline(FILTER_RMSD_MAX, color='red', linestyle='--', alpha=0.5)
        ax.axhline(FILTER_PAE_MAX,  color='red', linestyle='--', alpha=0.5)
        ax.set_xlabel('RMSD (Å)')
        ax.set_ylabel('pAE')
        ax.set_title(f'{target_name} — pAE vs RMSD')
        ax.legend()
        plt.tight_layout()

        scatter_path = output_dir / 'nb02_pae_vs_rmsd.png'
        plt.savefig(scatter_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Scatter plot saved: {scatter_path}')
else:
    print('No scores available to plot.')
